# 01 Data Preparation

This notebook documents the data preparation stage for the curated perovskite training set.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(repo_root)


/root/大创/perovskite-bandgap-ml-correction


In [2]:
import pandas as pd
from src.data_pipeline import load_training_set, sanitize_columns, build_feature_matrix
from src.polymorph_mapping import summarize_formula_counts, select_formula_representatives

df_raw = load_training_set(repo_root / 'data' / 'training_set_257.csv')
df = sanitize_columns(df_raw)
print('raw shape:', df_raw.shape)
print('sanitized shape:', df.shape)
df[['Formula', 'pretty_formula', 'E_g_Exp', 'band_gap']].head()


raw shape: (257, 293)
sanitized shape: (257, 293)


,Formula,pretty_formula,E_g_Exp,band_gap
0,CsPbBr3,CsPbBr3,2.333235,1.9528
1,CsPbBr3,CsPbBr3,2.333235,1.8178
2,CsPbBr3,CsPbBr3,2.333235,2.0087
3,CsPbI3,CsPbI3,1.895000,1.6364
4,AgGeO3,AgGeO3,0.000000,0.0000


In [3]:
summary = df['E_g_Exp'].describe()
metal_count = int((df['E_g_Exp'] <= 0.05).sum())
nonmetal_count = int((df['E_g_Exp'] > 0.05).sum())
print(summary)
print({'metal_or_zero_gap': metal_count, 'nonmetal': nonmetal_count})


count    257.000000
mean       0.942726
std        1.386334
min        0.000000
25%        0.000000
50%        0.000000
75%        2.000000
max        4.400000
Name: E_g_Exp, dtype: float64
{'metal_or_zero_gap': 142, 'nonmetal': 115}


In [4]:
drop_cols = ['Formula', 'E_g_Exp', 'Source', 'Priority', 'pretty_formula', 'Delta_E_g', 'is_metal_exp', 'target_delta']
X, feature_cols = build_feature_matrix(df, drop_cols)
print('feature matrix shape:', X.shape)
print('number of features:', len(feature_cols))
print(feature_cols[:20])


feature matrix shape: (257, 285)
number of features: 285
['Pt fraction', 'Hf fraction', 'X_site_electronegativity', 'Ga fraction', 'He fraction', 'avg anion electron affinity', 'F fraction', 'maximum EN difference', 'Sc fraction', 'N fraction', 'Yb fraction', 'Au fraction', 'Mo fraction', 'Li fraction', 'Mg fraction', 'As fraction', 'Ca fraction', 'Bk fraction', 'B fraction', 'range Electronegativity']


In [5]:
formula_counts = summarize_formula_counts(df_raw, formula_col='Formula')
formula_counts.head(10)


,Formula,count
0,LiAgF3,7
1,BaBiO3,6
2,CeAlO3,5
3,LaGaO3,5
4,SrSnO3,4
5,AgGeO3,4
6,AgNO3,4
7,YTiO3,3
8,Sr2FeMoO6,3
9,YCoO3,3


In [6]:
representatives = select_formula_representatives(df_raw, formula_col='Formula')
print('unique-formula representatives:', representatives.shape[0])
representatives[['Formula', 'pretty_formula', 'E_g_Exp', 'band_gap']].head()


unique-formula representatives: 183


,Formula,pretty_formula,E_g_Exp,band_gap
0,CsPbBr3,CsPbBr3,2.333235,1.9528
1,CsPbI3,CsPbI3,1.895000,1.6364
2,AgGeO3,AgGeO3,0.000000,0.0000
3,Ba2CoWO6,Ba2CoWO6,0.000000,0.0000
4,Ba2CuWO6,Ba2CuWO6,0.000000,0.0000
